# Medición 4/8/2026

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from utils.oscrigol_oil import oil_med, plotresults

In [ ]:
tdiv = 10e-6
tmed = 70.0e-6
Amp = "Judson"
lenteUS = -40e-3
lenteOA = 60e-3
OscCal = True

MDEPTH = 70000
acq = 1
Nmed = 10
save = True

# Opciones de comunicacion/adquisicion nuevas de oscrigol.py.
TRANSPORT = "socket"  # "socket" recomendado; "instr" usa VXI-11/INSTR
VISA_BACKEND = "pyvisa"  # "pyvisa" por defecto; "nivisa" si se quiere usar NI-VISA
DOWNLOAD_MODE = "fast"  # "fast" recomendado; "legacy" conserva el flujo anterior

# Parametros del modo fast. Mantenerlos visibles para poder ajustar estabilidad vs tiempo.
POLL_INTERVAL = 0.005
TRIGGER_TIMEOUT = 5.0
WAVEFORM_DELAY = 0.01
WAVEFORM_POINTS_TIMEOUT = 0.5
ARM_DELAY = 0.05
MAX_RETRIES = 2
MIN_VPP = None  # ejemplo: 0.05 para rechazar capturas con menos de 50 mVpp

# La energia laser que devuelve oil_med no tiene en cuenta divisor ni lentes.
# Las soluciones se revolvieron con agitador magnetico.


## Captura con la nueva biblioteca

La llamada sigue siendo compatible con la forma anterior, pero ahora las opciones de transporte, backend VISA, modo de descarga y reintentos quedan explicitas en la celda de parametros. Para una adquisicion valida se espera `MV.shape[1] == MDEPTH` y una senal con Vpp consistente.


In [ ]:
t, MV, E, T = oil_med(
    Nmed=Nmed,
    acq=acq,
    mdepth=MDEPTH,
    saveresults=save,
    transport=TRANSPORT,
    visa_backend=VISA_BACKEND,
    download_mode=DOWNLOAD_MODE,
    poll_interval=POLL_INTERVAL,
    trigger_timeout=TRIGGER_TIMEOUT,
    waveform_delay=WAVEFORM_DELAY,
    waveform_points_timeout=WAVEFORM_POINTS_TIMEOUT,
    arm_delay=ARM_DELAY,
    max_retries=MAX_RETRIES,
    min_vpp=MIN_VPP,
)

print(f"MV shape: {MV.shape}")
print(f"t samples: {t.size}")
print(f"Energy finite: {np.isfinite(E).sum()}/{E.size}")
print(f"Vpp min/median/max [mV]: {np.ptp(MV, axis=1).min()*1e3:.2f} / "
      f"{np.median(np.ptp(MV, axis=1))*1e3:.2f} / {np.ptp(MV, axis=1).max()*1e3:.2f}")


In [ ]:
i = 13
Nmeas = 14
MVp = np.mean(MV,axis=0)
Ep = np.mean(E)
if i == 0:
    MVn = np.zeros((Nmeas,len(MVp)))
MVn[i,:] = MVp/Ep
for j in range(i+1):
    plt.plot(t*1e6,MVn[j,:]-np.mean(MVn[j,:]),label='med'+str(j))
plt.grid(linestyle = '--')
plt.xlabel('time [us]'); plt.ylabel('Normalized amplitude (V/J)')
plt.legend()
plt.xlim([17, 23])
plt.ylim([-5, 10])

In [ ]:
for j in range(i+1):
    plt.plot(t*1e6,MVn[j,:]-np.mean(MVn[j,:]),label='med'+str(j))
plt.grid(linestyle = '--')
plt.xlabel('time [us]'); plt.ylabel('Normalized amplitude (V/J)')
plt.legend()
plt.xlim([2.5, 6])

In [ ]:
plotresults(t, MV, E, T)

In [ ]:
plt.figure()
for i in range(Nmed):
    plt.plot(t*1e6,MV[i,:]*1e3)#,label='avg = 1')
plt.grid(linestyle = '--')
plt.xlabel('time [us]'); plt.ylabel('Amplitude [mV]')
plt.xlim([10, 30])
#plt.ylim([-5, 5])
#plt.title('')
#plt.legend()